# 02_exploration_indeed

**Welcome to the Jungle** (recommandé) => Site français, peu protégé, riche en offres tech : await page.goto("https://www.welcometothejungle.com/fr/jobs?refinementList%5Boffice_country_codes%5D%5B%5D=FR&query=data+engineer")

In [1]:
from pprint import pprint
import json
from rich.tree import Tree
from rich import print as rprint

In [2]:
def json_vers_arbre(data, arbre, prefixe=""):
    """
    Construit récursivement un arbre Rich
    à partir d'un dictionnaire JSON.
    """
    if isinstance(data, dict):
        for cle, valeur in data.items():
            if isinstance(valeur, dict):
                # Sous-dictionnaire → nouveau nœud
                noeud = arbre.add(f"[bold cyan]{cle}[/] 📁")
                json_vers_arbre(valeur, noeud)

            elif isinstance(valeur, list):
                # Tableau → nouveau nœud avec indication du type
                noeud = arbre.add(f"[bold yellow]{cle}[/] 📋 [dim]({len(valeur)} éléments)[/]")
                if valeur and isinstance(valeur[0], dict):
                    json_vers_arbre(valeur[0], noeud)

            else:
                # Valeur simple → feuille
                type_valeur = type(valeur).__name__
                arbre.add(f"[green]{cle}[/] [dim]({type_valeur})[/]")


def afficher_structure_json(data, titre="Structure JSON"):
    """
    Affiche la structure d'un JSON sous forme d'arbre visuel.
    """
    arbre = Tree(f"[bold magenta]{titre}[/]")
    json_vers_arbre(data, arbre)
    rprint(arbre)

In [3]:
from playwright.async_api import async_playwright

'''Ce script simule un vrai navigateur Chrome pour accéder au site Welcome to the Jungle, 
puis interroge directement le moteur de recherche interne du site (Algolia) pour récupérer
les offres d'emploi Data Engineer en France. L'astuce centrale est d'utiliser le navigateur 
comme intermédiaire de confiance — Algolia refuse les requêtes venant d'un script Python externe, 
mais accepte celles venant d'un navigateur sur le domaine de WTTJ. Le script récupère les offres
page par page et les accumule dans une liste.'''

# Fonction principale — nb_pages contrôle combien de pages on récupère
# (20 offres par page, donc nb_pages=3 → 60 offres maximum)
async def scraper_wttj(nb_pages=3):

    # Liste qui accumulera toutes les offres récupérées
    toutes_offres = []

    # Démarrage de Playwright — le gestionnaire de navigateur
    async with async_playwright() as p:

        # Lancement d'un navigateur Chromium
        # headless=True → pas de fenêtre visible (mode silencieux)
        # disable-blink-features=AutomationControlled → masque le fait
        # que Chrome est piloté par un script (contourne certaines détections)
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )

        # Création d'un contexte de navigation — comme un profil de navigateur
        # user_agent → on se fait passer pour un vrai Chrome sur Windows
        # viewport  → résolution d'écran standard d'un vrai utilisateur
        # locale    → langue française pour recevoir le bon contenu
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1920, "height": 1080},
            locale="fr-FR",
        )

        # Injection d'un script JavaScript exécuté avant chaque page
        # navigator.webdriver est une propriété que les sites utilisent
        # pour détecter les navigateurs automatisés — on la masque
        # en lui faisant retourner "undefined" au lieu de "true"
        await context.add_init_script("""
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            });
        """)

        # Ouverture d'un nouvel onglet dans le contexte créé
        page = await context.new_page()

        # Navigation vers la page de recherche WTTJ
        # wait_until="domcontentloaded" → on attend que le HTML de base
        # soit chargé, sans attendre tous les appels réseau
        # (networkidle bloquerait indéfiniment sur ce site)
        await page.goto(
            "https://www.welcometothejungle.com/fr/jobs?"
            "refinementList%5Boffice_country_codes%5D%5B%5D=FR"
            "&query=data+engineer",
            wait_until="domcontentloaded"
        )

        # Pause de 2 secondes pour laisser le JavaScript s'initialiser
        # et établir la session avec Algolia
        await page.wait_for_timeout(2000)

        # Boucle sur chaque page de résultats
        for numero_page in range(nb_pages):
            print(f"Page {numero_page + 1}/{nb_pages}...")

            # Exécution d'un appel fetch() directement dans le navigateur
            # C'est la clé du fonctionnement : on appelle l'API Algolia
            # DEPUIS le navigateur qui est sur le domaine WTTJ
            # → Algolia accepte la requête car elle vient du bon domaine
            # f""" ... """ permet d'injecter la variable Python numero_page
            # dans le code JavaScript avec {numero_page}
            resultats = await page.evaluate(f"""
                async () => {{
                    const response = await fetch(
                        "https://CSEKHVMS53-dsn.algolia.net/1/indexes/wttj_jobs_production_fr/query",
                        {{
                            method: "POST",
                            headers: {{
                                // Identifiant de l'application Algolia de WTTJ
                                "X-Algolia-Application-Id": "CSEKHVMS53",
                                // Clé publique extraite des requêtes du navigateur
                                "X-Algolia-API-Key": "4bd8f6215d0cc52b26430765769e65a0",
                                "Content-Type": "application/json"
                            }},
                            body: JSON.stringify({{
                                query: "data engineer",  // mots recherchés
                                hitsPerPage: 20,         // 20 offres par page
                                page: {numero_page}      // numéro de page (0, 1, 2...)
                            }})
                        }}
                    );
                    // Conversion de la réponse HTTP en objet JSON Python
                    return await response.json();
                }}
            """)

            # Extraction des offres (hits) et du nombre total depuis la réponse
            hits    = resultats.get("hits", [])     # liste des offres de cette page
            nb_hits = resultats.get("nbHits", 0)    # nombre total d'offres disponibles

            print(f"  → {len(hits)} hits, {nb_hits} total")

            # Ajout des offres de cette page à la liste globale
            toutes_offres.extend(hits)

            # Pause de 0.5 seconde entre chaque requête
            # pour ne pas surcharger le serveur et éviter la détection
            await page.wait_for_timeout(500)

        # Fermeture du navigateur — libération des ressources
        await browser.close()

    # Retourne la liste complète de toutes les offres récupérées
    return toutes_offres

In [4]:
if __name__ == "__main__":
    
    offres = await scraper_wttj(nb_pages=1)
    print("offres ok")

    print(f"\n✅ {len(offres)} offres récupérées")
    print("Nombre offres ok")

    if offres:
        print("\nClés disponibles :")
        print(list(offres[0].keys()))
    print("Clés offres ok")

    # Afficher 
    afficher_structure_json(offres[0], "Offre Welcome to the Jungle")
    pprint(offres[0])
    print("FIN")

Page 1/1...
  → 20 hits, 1588 total
offres ok

✅ 20 offres récupérées
Nombre offres ok

Clés disponibles :
['published_at_timestamp', 'sectors', 'contract_duration_maximum', 'rank_group_1', 'benefits', 'key_missions', 'rank_group_2', 'profile_ranking', '_geoloc', 'education_level', 'source_stage', 'salary_minimum', 'published_at_date', 'has_education_level', 'language', 'rank_group_3', 'slug', 'experience_level_minimum', 'contract_duration_minimum', 'has_experience_level_minimum', 'has_benefits', 'organization', 'remote', 'summary', 'has_remote', 'has_contract_duration', 'salary_maximum', 'offices', 'is_boosted', 'profile', 'organization_score', 'name', 'has_salary_yearly_minimum', 'reference', 'salary_period', 'new_profession', 'wk_reference', 'published_at', 'contract_type', 'salary_currency', 'salary_yearly_minimum', 'objectID', '_highlightResult']
Clés offres ok


Offre Welcome to the Jungle
├── published_at_timestamp (int)
├── sectors 📋 (2 éléments)
│   ├── name (str)
│   ├── reference (str)
│   ├── parent_name (str)
│   └── parent_reference (str)
├── contract_duration_maximum (NoneType)
├── rank_group_1 (int)
├── benefits 📋 (5 éléments)
├── key_missions 📋 (3 éléments)
├── rank_group_2 (int)
├── profile_ranking (int)
├── _geoloc 📋 (1 éléments)
│   ├── lat (float)
│   └── lng (float)
├── education_level (NoneType)
├── source_stage (str)
├── salary_minimum (int)
├── published_at_date (str)
├── has_education_level (bool)
├── language (str)
├── rank_group_3 (int)
├── slug (str)
├── experience_level_minimum (int)
├── contract_duration_minimum (NoneType)
├── has_experience_level_minimum (bool)
├── has_benefits (bool)
├── organization 📁
│   ├── name (str)
│   ├── description (str)
│   ├── reference (str)
│   ├── labels 📋 (0 éléments)
│   ├── summary (str)
│   ├── profile_type (str)
│   ├── slug (str)
│   ├── logo 📁
│   │   ├── url (str)
│   │   └── thumb 📁
│   │       └── url (str)
│   ├── nb_employees (int)
│   ├── creation_year (int)
│   ├── cover_image 📁
│   │   ├── small 📁
│   │   │   └── url (str)
│   │   ├── url (str)
│   │   ├── medium 📁
│   │   │   └── url (str)
│   │   ├── large 📁
│   │   │   └── url (str)
│   │   └── social 📁
│   │       └── url (str)
│   ├── profile_ranking (int)
│   ├── commitments 📋 (0 éléments)
│   └── equality_index (NoneType)
├── remote (str)
├── summary (str)
├── has_remote (bool)
├── has_contract_duration (bool)
├── salary_maximum (int)
├── offices 📋 (1 éléments)
│   ├── state (str)
│   ├── city (str)
│   ├── country (str)
│   ├── country_code (str)
│   ├── district (str)
│   ├── local_district (str)
│   ├── local_city (str)
│   └── local_state (str)
├── is_boosted (bool)
├── profile (str)
├── organization_score (int)
├── name (str)
├── has_salary_yearly_minimum (bool)
├── reference (str)
├── salary_period (str)
├── new_profession 📁
│   ├── sub_category_reference (str)
│   ├── sub_category_name (str)
│   ├── category_reference (str)
│   ├── category_name (str)
│   ├── pivot_name (str)
│   └── pivot_reference (str)
├── wk_reference (str)
├── published_at (str)
├── contract_type (str)
├── salary_currency (str)
├── salary_yearly_minimum (int)
├── objectID (str)
└── _highlightResult 📁
    ├── organization 📁
    │   └── name 📁
    │       ├── value (str)
    │       ├── matchLevel (str)
    │       └── matchedWords 📋 (0 éléments)
    ├── summary 📁
    │   ├── value (str)
    │   ├── matchLevel (str)
    │   ├── fullyHighlighted (bool)
    │   └── matchedWords 📋 (1 éléments)
    ├── offices 📋 (1 éléments)
    │   ├── state 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── city 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── country 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── country_code 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── district 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── local_district 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── local_city 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   └── local_state 📁
    │       ├── value (str)
    │       ├── matchLevel (str)
    │       └── matchedWords 📋 (0 éléments)
    ├── profile 📁
    │   ├── value (str)
    │   ├── matchLevel (str)
    │   ├── fullyHighlighted (bool)
    │   └── matchedWords 📋 (1 éléments)
    ├── name 📁
    │   ├── value (str)
    │   ├── matchLevel (str)
    │   ├── fullyHighlighted (bool)
    │   └── matchedWords 📋 (1 éléments)
    └── new_profession 📁
        ├── sub_category_reference 📁
 

{'_geoloc': [{'lat': 48.88525, 'lng': 2.31627}],
 '_highlightResult': {'name': {'fullyHighlighted': False,
                               'matchLevel': 'partial',
                               'matchedWords': ['engineer'],
                               'value': 'Senior Machine Learning '
                                        '<em>Engineer</em>'},
                      'new_profession': {'category_name': {'matchLevel': 'none',
                                                           'matchedWords': [],
                                                           'value': 'Technologie '
                                                                    'et '
                                                                    'ingénierie'},
                                         'pivot_name': {'matchLevel': 'none',
                                                        'matchedWords': [],
                                                        'value': 'Ingénieur en '
            